In [1]:
import sys
import os

project_root = os.path.dirname(os.path.dirname(os.getcwd()))
src_path = os.path.join(project_root, 'src')
sys.path.append(src_path)



import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from algorithms.collaborative_filtering import UserBasedCF, ItemBasedCF
from algorithms.matrix_factorization import BasicMatrixFactorization, SVDMatrixFactorization
from algorithms.content_based import GenreBasedRecommender, DemographicBasedRecommender
from algorithms.hybrid import HybridRecommender
from algorithms.baselines import GlobalAverageRecommender, UserAverageRecommender, MovieAverageRecommender, BiasRecommender


ratings_df = pd.read_csv('../../data/ratings_processed.csv')  
movies_df = pd.read_csv('../../data/movies_processed.csv')  
users_df = pd.read_csv('../../data/users_processed.csv')


In [2]:
def draw_plots(rmse_values, mae_values, indexes,algorithm_name):
    figure = plt.figure(figsize=(20, 10))
    ax1 = figure.add_subplot(1,2,1)
    ax1.plot(indexes, rmse_values, label=algorithm_name,color = 'red',marker = 'o')
    ax1.set_xlabel('Number of latent factors')
    ax1.set_ylabel('RMSE')
    ax1.legend()
    ax2 = figure.add_subplot(1,2,2)
    ax2.plot(indexes, mae_values, label=algorithm_name,color = 'blue',marker = 'o')
    ax2.set_xlabel('Number of latent factors')
    ax2.set_ylabel('MAE')
    ax2.legend()
    plt.show()

In [3]:

def tune_baselines_models(algorithm_name : str,train_df : pd.DataFrame ,test_dict : dict,actual_values : list):
    if algorithm_name == 'GlobalAverageRecommender':
        model = GlobalAverageRecommender() 
    elif algorithm_name == 'UserAverageRecommender':
        model = UserAverageRecommender()
    elif algorithm_name == 'MovieAverageRecommender':
        model = MovieAverageRecommender()
    elif algorithm_name == 'BiasRecommender':
        model = BiasRecommender()
    else:
        raise ValueError(f"Algorithm {algorithm_name} not found")
    model.fit(train_df)
    predictions = []
    for user in test_dict.keys():
        predictions_per_user = model.predict_for_user(user,test_dict[user])
        for prediction in predictions_per_user:
            predictions.append(prediction[0])
    predictions = np.array(predictions)
    actual_values = np.array(actual_values)
    rmse = np.sqrt(np.mean((predictions - actual_values)**2))
    mae = np.mean(np.abs(predictions - actual_values))
    print(f"RMSE for {algorithm_name}: {round(rmse,2)}")
    print(f"MAE for {algorithm_name}: {round(mae,2)}")

In [4]:
def tune_collaborative_filtering_models(algorithm_name : str,train_df : pd.DataFrame ,test_dict : dict,actual_values : list):
    print(f"Tuning {algorithm_name}...")
    print("--------------------------------")
    k_values = [10,20,30,40,50]
    rmse_values = []
    mae_values = []
    best_rmse = float('inf')
    best_mae = float('inf')
    best_params = None
    for k in k_values:
        if algorithm_name == 'UserBasedCF':
            model = UserBasedCF()
        elif algorithm_name == 'ItemBasedCF':
            model = ItemBasedCF()
        else:
            raise ValueError(f"Algorithm {algorithm_name} not found")
        print(f"Tuning {algorithm_name} with k = {k}...")
        model.k = k
        model.fit(train_df)
        predictions = []
        for user in test_dict.keys():
            print(f"Predicting for user {user}...")
            predictions_per_user = model.predict_for_user(user,test_dict[user])
            for prediction in predictions_per_user:
                print(f"Prediction: {prediction}")
                predictions.append(prediction[0])
        predictions = np.array(predictions)
        actual_values = np.array(actual_values)
        rmse = np.sqrt(np.mean((predictions - actual_values)**2))
        mae = np.mean(np.abs(predictions - actual_values))
        if rmse < best_rmse:
            best_rmse = rmse
            best_params = k
        if mae < best_mae:
            best_mae = mae
          
        rmse_values.append(rmse)
        mae_values.append(mae)
    print(f"RMSE for {algorithm_name}: {round(best_rmse,2)}")
    print(f"MAE for {algorithm_name}: {round(best_mae,2)}")
    print(f"Best RMSE parameters for {algorithm_name}: {best_params}")
    draw_plots(rmse_values, mae_values, range(len(k_values)),algorithm_name)


In [5]:
def tune_content_based_models(algorithm_name : str,train_df : pd.DataFrame ,test_dict : dict,actual_values : list):
    best_rmse = float('inf')
    best_mae = float('inf')
    best_params = None
    if algorithm_name == 'GenreBasedRecommender':
        model = GenreBasedRecommender()
        model.fit(train_df,movies_df)
        predictions = []
        for user in test_dict.keys():
            predictions_per_user = model.predict_for_user(user,test_dict[user])
            for prediction in predictions_per_user:
                predictions.append(prediction[0])
        predictions = np.array(predictions)
        actual_values = np.array(actual_values)
        best_rmse = np.sqrt(np.mean((predictions - actual_values)**2))
        best_mae = np.mean(np.abs(predictions - actual_values))

    elif algorithm_name == 'DemographicBasedRecommender':
        model = DemographicBasedRecommender()
        k_values = [10,20,30,40,50]
        rmse_values = []
        mae_values = []
        for k in k_values:
            model.k = k
            model.fit(train_df,users_df)
            predictions = []
            for user in test_dict.keys():
                predictions_per_user = model.predict_for_user(user,test_dict[user])
                for prediction in predictions_per_user:
                    predictions.append(prediction[0])
            predictions = np.array(predictions)
            actual_values = np.array(actual_values)
            rmse = np.sqrt(np.mean((predictions - actual_values)**2))
            mae = np.mean(np.abs(predictions - actual_values))
            if rmse < best_rmse:
                best_rmse = rmse
                best_params = k
            if mae < best_mae:
                best_mae = mae
            rmse_values.append(rmse)
            mae_values.append(mae)
    else:
        raise ValueError(f"Algorithm {algorithm_name} not found")
    print(f"RMSE for {algorithm_name}: {round(best_rmse,2)}")
    print(f"MAE for {algorithm_name}: {round(best_mae,2)}")
    if algorithm_name == 'DemographicBasedRecommender':
        print(f"Best RMSE parameters for {algorithm_name}: {best_params}")
        draw_plots(rmse_values, mae_values, range(len(k_values)),algorithm_name)

In [6]:
from sklearn.model_selection import ParameterGrid
def tune_matrix_factorization_models(algorithm_name : str,train_df : pd.DataFrame ,test_dict : dict,actual_values : list):
    best_rmse = float('inf')
    best_mae = float('inf')
    best_params = None
    rmse_values = []
    mae_values = []
    if algorithm_name == 'SVDMatrixFactorization':
        model = SVDMatrixFactorization()
        model.fit(train_df)
        predictions = []
        for user in test_dict.keys():
            predictions_per_user = model.predict_for_user(user,test_dict[user])
            for prediction in predictions_per_user:
                predictions.append(prediction[0])
        predictions = np.array(predictions)
        actual_values = np.array(actual_values)
        best_rmse = np.sqrt(np.mean((predictions - actual_values)**2))
        best_mae = np.mean(np.abs(predictions - actual_values))
    elif algorithm_name == 'BasicMatrixFactorization':
        model = BasicMatrixFactorization()
        param_grid = {
            'k': [10,20,30,40,50],
            'epochs': [100,200,300,400,500],
            'learning_rate': [0.001,0.01,0.1],
            'reg': [0.001,0.01,0.1]
        }
        parametrs = ParameterGrid(param_grid)
        for params in parametrs:
            model.k = params['k']
            model.epochs = params['epochs']
            model.learning_rate = params['learning_rate']
            model.reg = params['reg']
            model.fit(train_df)
            predictions = []
            for user in test_dict.keys():
                predictions_per_user = model.predict_for_user(user,test_dict[user])
                for prediction in predictions_per_user:
                    predictions.append(prediction[0])
            predictions = np.array(predictions)
            actual_values = np.array(actual_values)
            rmse = np.sqrt(np.mean((predictions - actual_values)**2))
            mae = np.mean(np.abs(predictions - actual_values))
            if rmse < best_rmse:
                best_rmse = rmse
                best_params = params
            if mae < best_mae:
                best_mae = mae
            rmse_values.append(rmse)
            mae_values.append(mae)
    else:
        raise ValueError(f"Algorithm {algorithm_name} not found")
    print(f"RMSE for {algorithm_name}: {round(best_rmse,2)}")
    print(f"MAE for {algorithm_name}: {round(best_mae,2)}")
    if algorithm_name == 'BasicMatrixFactorization':
        print(f"Best RMSE parameters for {algorithm_name}: {best_params}")
        draw_plots(rmse_values, mae_values, range(len(parametrs)),algorithm_name)

        

In [7]:
def prepare_data_for_tuning(missing_percentage: int):
    test_size = missing_percentage / 100
    test_df = ratings_df.sample(frac=test_size, random_state=42)
    train_df = ratings_df.drop(test_df.index)
    
    users_to_predict = {}
    actual_ratings = []
    
    for _, row in test_df.iterrows():
        user = row['userId']
        movie = row['movieId']
        rating = row['rating']
        
        if user not in users_to_predict:
            users_to_predict[user] = []
        users_to_predict[user].append(movie)
        actual_ratings.append(rating)
    
    return users_to_predict, actual_ratings, train_df

In [ ]:
'''Оптимизация процесса выбора модели : сделай мальникие семплы по 500 пользователей, 
перепеши классы на векторных операциях, сделай тюнинг на маленьких семплах каждой отедльной 
модели потом бери лучше параметры и сделай тюнинг гибродной системы с этими параметрами, 
скорее всего гибридная система будет работать лучше чем каждая из моделей по отдельности, после
возьми эту гибридную систему и сделай тюнинг на всех данных'''

test_dict,actual_values,train_df = prepare_data_for_tuning(20)










In [9]:
tune_baselines_models('GlobalAverageRecommender',train_df,test_dict,actual_values)
tune_baselines_models('UserAverageRecommender',train_df,test_dict,actual_values)
tune_baselines_models('MovieAverageRecommender',train_df,test_dict,actual_values)
tune_baselines_models('BiasRecommender',train_df,test_dict,actual_values)

RMSE for GlobalAverageRecommender: 1.12
MAE for GlobalAverageRecommender: 0.94
RMSE for UserAverageRecommender: 1.2
MAE for UserAverageRecommender: 0.97


KeyError: 286

In [10]:
tune_collaborative_filtering_models('UserBasedCF',train_df,test_dict,actual_values)
tune_collaborative_filtering_models('ItemBasedCF',train_df,test_dict,actual_values)

Tuning UserBasedCF...
--------------------------------
Tuning UserBasedCF with k = 10...
Predicting for user 5412...
Prediction: (3.850467289719626, 2683)
Prediction: (3.850467289719626, 2745)
Prediction: (3.850467289719626, 1690)
Prediction: (3.850467289719626, 3452)
Prediction: (3.850467289719626, 1196)
Prediction: (3.850467289719626, 1688)
Prediction: (3.850467289719626, 112)
Prediction: (3.850467289719626, 3082)
Prediction: (3.850467289719626, 539)
Prediction: (3.850467289719626, 1136)
Prediction: (3.850467289719626, 1453)
Prediction: (3.850467289719626, 1210)
Prediction: (3.850467289719626, 1293)
Prediction: (3.850467289719626, 589)
Prediction: (3.850467289719626, 2161)
Prediction: (3.850467289719626, 464)
Prediction: (3.850467289719626, 1918)
Prediction: (3.850467289719626, 2712)
Prediction: (3.850467289719626, 344)
Prediction: (3.850467289719626, 2997)
Prediction: (3.850467289719626, 1687)
Prediction: (3.850467289719626, 1275)
Prediction: (3.850467289719626, 1258)
Prediction: (3

KeyboardInterrupt: 

In [ ]:
tune_content_based_models('GenreBasedRecommender',train_df,test_dict,actual_values)
tune_content_based_models('DemographicBasedRecommender',train_df,test_dict,actual_values)

In [ ]:
tune_matrix_factorization_models('BasicMatrixFactorization',train_df,test_dict,actual_values)
tune_matrix_factorization_models('SVDMatrixFactorization',train_df,test_dict,actual_values)